In [1]:
from pathlib import Path
import gc
import json
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


PROJECT_ROOT = Path(
    "/home/geonug/kdt-linux/ML"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v4/training/train_v4b2.parquet"
)

VALID_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v4/validation/valid_v2_top40.parquet"
)

TOP40_PATH = (
    PROJECT_ROOT
    / "results/feature_selection/top40_features.csv"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models/network"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results/dnn_top40_v4"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


TOP40_FEATURES = (
    pd.read_csv(TOP40_PATH)["feature"]
    .astype(str)
    .tolist()
)

assert len(TOP40_FEATURES) == 40

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Train:", TRAIN_PATH)
print("Valid:", VALID_PATH)
print("Features:", len(TOP40_FEATURES))

Torch: 2.14.1+cu130
CUDA: False
Train: /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/training/train_v4b2.parquet
Valid: /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/validation/valid_v2_top40.parquet
Features: 40


In [2]:
from sklearn.preprocessing import StandardScaler
import joblib


SCALER_PATH = (
    MODEL_DIR
    / "dnn_top40_v4_scaler.joblib"
)

scaler = StandardScaler()


parquet_file = pq.ParquetFile(
    TRAIN_PATH
)

rows_seen = 0


for batch_id, batch in enumerate(
    parquet_file.iter_batches(
        batch_size=100_000,
        columns=TOP40_FEATURES,
    ),
    start=1,
):

    X = (
        batch
        .to_pandas()
        .to_numpy(
            dtype=np.float32,
            copy=False,
        )
    )

    # inf 방어
    X[~np.isfinite(X)] = np.nan

    # StandardScaler는 NaN을 무시할 수 있지만
    # DNN 입력에는 NaN을 넣으면 안 되므로
    # 우선 scaler 통계만 계산
    scaler.partial_fit(X)

    rows_seen += len(X)

    print(
        f"[scaler {batch_id:02d}] "
        f"rows={rows_seen:,}"
    )

    del X, batch
    gc.collect()


joblib.dump(
    scaler,
    SCALER_PATH,
)

print("Scaler saved:", SCALER_PATH)
print("Rows:", rows_seen)

[scaler 01] rows=100,000
[scaler 02] rows=200,000
[scaler 03] rows=300,000
[scaler 04] rows=400,000
[scaler 05] rows=500,000
[scaler 06] rows=600,000
[scaler 07] rows=700,000
[scaler 08] rows=800,000
[scaler 09] rows=900,000
[scaler 10] rows=1,000,000
[scaler 11] rows=1,100,000
[scaler 12] rows=1,200,000
[scaler 13] rows=1,300,000
[scaler 14] rows=1,400,000
[scaler 15] rows=1,500,000
[scaler 16] rows=1,600,000
[scaler 17] rows=1,700,000
[scaler 18] rows=1,800,000
[scaler 19] rows=1,900,000
[scaler 20] rows=1,946,494
Scaler saved: /home/geonug/kdt-linux/ML/models/network/dnn_top40_v4_scaler.joblib
Rows: 1946494


In [3]:
def preprocess_batch(
    X,
    scaler,
):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    X[~np.isfinite(X)] = np.nan

    means = scaler.mean_.astype(
        np.float32
    )

    nan_rows, nan_cols = np.where(
        np.isnan(X)
    )

    if len(nan_rows) > 0:
        X[nan_rows, nan_cols] = means[nan_cols]

    X = scaler.transform(
        X
    ).astype(
        np.float32,
        copy=False,
    )

    return X

In [4]:
class NetworkThreatDNN(nn.Module):

    def __init__(
        self,
        input_dim=40,
        num_classes=15,
    ):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                128,
            ),

            nn.BatchNorm1d(
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                0.20
            ),


            nn.Linear(
                128,
                128,
            ),

            nn.BatchNorm1d(
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                0.20
            ),


            nn.Linear(
                128,
                64,
            ),

            nn.ReLU(),


            nn.Linear(
                64,
                num_classes,
            ),
        )


    def forward(
        self,
        x,
    ):
        return self.network(x)

In [5]:
model = NetworkThreatDNN()

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Parameters: {num_params:,}"
)

print(
    f"FP32 weight size ≈ "
    f"{num_params * 4 / 1024**2:.3f} MB"
)

Parameters: 31,503
FP32 weight size ≈ 0.120 MB


In [6]:
TRAIN_COLUMNS = (
    TOP40_FEATURES
    + [
        "label_id",
        "sample_weight",
    ]
)


train_df = pd.read_parquet(
    TRAIN_PATH,
    columns=TRAIN_COLUMNS,
)

print(
    "Loaded:",
    train_df.shape
)

print(
    "DataFrame memory:",
    f"{train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

Loaded: (1946494, 42)
DataFrame memory: 612.59 MB


In [7]:
X_train = (
    train_df[
        TOP40_FEATURES
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)

y_train = (
    train_df[
        "label_id"
    ]
    .to_numpy(
        dtype=np.int64,
        copy=True,
    )
)

w_train = (
    train_df[
        "sample_weight"
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)


del train_df
gc.collect()


X_train = preprocess_batch(
    X_train,
    scaler,
)


print(
    "X:",
    X_train.shape,
    X_train.dtype,
)

print(
    "y:",
    y_train.shape,
)

print(
    "weight:",
    w_train.shape,
)

print(
    "finite:",
    np.isfinite(
        X_train
    ).all()
)

X: (1946494, 40) float32
y: (1946494,)
weight: (1946494,)
finite: True


In [8]:
X_tensor = torch.from_numpy(
    X_train
)

y_tensor = torch.from_numpy(
    y_train
)

w_tensor = torch.from_numpy(
    w_train
)


train_dataset = torch.utils.data.TensorDataset(
    X_tensor,
    y_tensor,
    w_tensor,
)


train_loader = DataLoader(
    train_dataset,

    batch_size=4096,

    shuffle=True,

    num_workers=0,

    pin_memory=False,
)

In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


model = NetworkThreatDNN().to(
    device
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)


criterion = nn.CrossEntropyLoss(
    reduction="none"
)

Device: cpu


In [10]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
):

    model.train()

    total_loss = 0.0
    total_weight = 0.0

    start = time.perf_counter()


    for batch_id, (
        X,
        y,
        weights,
    ) in enumerate(
        loader,
        start=1,
    ):

        X = X.to(
            device,
            non_blocking=False,
        )

        y = y.to(
            device,
            non_blocking=False,
        )

        weights = weights.to(
            device,
            non_blocking=False,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(X)

        losses = criterion(
            logits,
            y,
        )


        loss = (
            losses
            * weights
        ).sum() / weights.sum()


        loss.backward()

        optimizer.step()


        batch_weight = (
            weights.sum().item()
        )

        total_loss += (
            loss.item()
            * batch_weight
        )

        total_weight += (
            batch_weight
        )


        if batch_id % 100 == 0:

            print(
                f"batch={batch_id:04d} "
                f"loss="
                f"{total_loss / total_weight:.6f}"
            )


    elapsed = (
        time.perf_counter()
        - start
    )


    return (
        total_loss
        / total_weight,
        elapsed,
    )

In [11]:
loss, elapsed = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    device,
)


print()
print(
    "=== DNN EPOCH 1 ==="
)

print(
    f"Loss : {loss:.6f}"
)

print(
    f"Time : {elapsed:.1f}s"
)

batch=0100 loss=0.618162
batch=0200 loss=0.422433
batch=0300 loss=0.341745
batch=0400 loss=0.298429

=== DNN EPOCH 1 ===
Loss : 0.276536
Time : 32.0s


In [12]:
DNN_CHECKPOINT = (
    MODEL_DIR
    / "dnn_top40_v4_epoch01.pt"
)


torch.save(
    {
        "epoch": 1,
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "top40_features":
            TOP40_FEATURES,

        "train_loss":
            loss,
    },
    DNN_CHECKPOINT,
)


print(
    "Saved:",
    DNN_CHECKPOINT
)

Saved: /home/geonug/kdt-linux/ML/models/network/dnn_top40_v4_epoch01.pt


In [13]:
import duckdb

DEV_VALID_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v4/validation/dnn_dev_valid.parquet"
)

con = duckdb.connect()

con.execute("SET memory_limit='2GB'")
con.execute("SET threads=2")
con.execute("SET preserve_insertion_order=false")

if not DEV_VALID_PATH.exists():

    parts = []

    for label_id in range(15):

        part = f"""
        (
            SELECT
                {", ".join('"' + f.replace('"', '""') + '"' for f in TOP40_FEATURES)},
                label_id
            FROM read_parquet('{VALID_PATH}')
            WHERE label_id = {label_id}
            LIMIT 5000
        )
        """

        parts.append(part)

    sql = "\nUNION ALL\n".join(parts)

    con.execute(
        f"""
        COPY (
            {sql}
        )
        TO '{DEV_VALID_PATH}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print("[DONE]", DEV_VALID_PATH)

else:
    print("[SKIP]", DEV_VALID_PATH)


print(
    con.execute(
        f"""
        SELECT label_id, COUNT(*) AS rows
        FROM read_parquet('{DEV_VALID_PATH}')
        GROUP BY label_id
        ORDER BY label_id
        """
    ).df().to_string(index=False)
)

[DONE] /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/validation/dnn_dev_valid.parquet
 label_id  rows
        0  5000
        1  5000
        2    98
        3    28
        4  5000
        5   268
        6  5000
        7  5000
        8  5000
        9  5000
       10  1611
       11  5000
       12  5000
       13    13
       14  5000


In [14]:
valid_df = pd.read_parquet(
    DEV_VALID_PATH,
    columns=TOP40_FEATURES + ["label_id"],
)

X_valid = valid_df[
    TOP40_FEATURES
].to_numpy(
    dtype=np.float32,
    copy=True,
)

y_valid = valid_df[
    "label_id"
].to_numpy(
    dtype=np.int64,
    copy=True,
)

del valid_df
gc.collect()


X_valid = preprocess_batch(
    X_valid,
    scaler,
)


X_valid_tensor = torch.from_numpy(
    X_valid
)

y_valid_tensor = torch.from_numpy(
    y_valid
)


valid_dataset = torch.utils.data.TensorDataset(
    X_valid_tensor,
    y_valid_tensor,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8192,
    shuffle=False,
    num_workers=0,
)

print("Validation:", len(valid_dataset))
print("finite:", np.isfinite(X_valid).all())

Validation: 52018
finite: True


In [15]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)


@torch.no_grad()
def evaluate_dnn(
    model,
    loader,
    device,
):

    model.eval()

    y_true_all = []
    y_pred_all = []


    for X, y in loader:

        X = X.to(
            device,
            non_blocking=False,
        )

        logits = model(X)

        pred = torch.argmax(
            logits,
            dim=1,
        )

        y_true_all.append(
            y.numpy()
        )

        y_pred_all.append(
            pred.cpu().numpy()
        )


    y_true = np.concatenate(
        y_true_all
    )

    y_pred = np.concatenate(
        y_pred_all
    )


    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

In [16]:
epoch1_metrics = evaluate_dnn(
    model,
    valid_loader,
    device,
)

print("=== EPOCH 1 DEV VALID ===")

for key, value in epoch1_metrics.items():
    print(
        f"{key:16s}: {value:.6f}"
    )

=== EPOCH 1 DEV VALID ===
accuracy        : 0.607867
macro_precision : 0.524753
macro_recall    : 0.452313
macro_f1        : 0.427476
weighted_f1     : 0.563709


In [17]:
MAX_EPOCHS = 15
PATIENCE = 3

BEST_MODEL_PATH = (
    MODEL_DIR
    / "dnn_top40_v4_best.pt"
)

HISTORY_PATH = (
    RESULT_DIR
    / "training_history.csv"
)


best_macro_f1 = (
    epoch1_metrics["macro_f1"]
)

best_epoch = 1

patience_count = 0


history = [
    {
        "epoch": 1,
        "train_loss": loss,
        **epoch1_metrics,
    }
]


# epoch 1을 현재 best로 저장
torch.save(
    {
        "epoch": 1,
        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "top40_features":
            TOP40_FEATURES,

        "scaler_path":
            str(SCALER_PATH),

        "metrics":
            epoch1_metrics,
    },
    BEST_MODEL_PATH,
)


for epoch in range(
    2,
    MAX_EPOCHS + 1,
):

    print()
    print(
        f"========== EPOCH {epoch} =========="
    )


    train_loss, train_time = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
    )


    metrics = evaluate_dnn(
        model,
        valid_loader,
        device,
    )


    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_seconds": train_time,
        **metrics,
    })


    print(
        f"Train loss      : {train_loss:.6f}"
    )

    print(
        f"Accuracy        : {metrics['accuracy']:.6f}"
    )

    print(
        f"Macro Precision : {metrics['macro_precision']:.6f}"
    )

    print(
        f"Macro Recall    : {metrics['macro_recall']:.6f}"
    )

    print(
        f"Macro F1        : {metrics['macro_f1']:.6f}"
    )

    print(
        f"Weighted F1     : {metrics['weighted_f1']:.6f}"
    )


    # ----------------------------------------
    # best checkpoint
    # ----------------------------------------

    if (
        metrics["macro_f1"]
        > best_macro_f1
    ):

        best_macro_f1 = (
            metrics["macro_f1"]
        )

        best_epoch = epoch

        patience_count = 0


        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "top40_features":
                    TOP40_FEATURES,

                "scaler_path":
                    str(SCALER_PATH),

                "metrics":
                    metrics,
            },
            BEST_MODEL_PATH,
        )


        print(
            f"[BEST] epoch={epoch} "
            f"macro_f1={best_macro_f1:.6f}"
        )


    else:

        patience_count += 1

        print(
            f"[NO IMPROVEMENT] "
            f"{patience_count}/{PATIENCE}"
        )


    # 매 epoch history 저장
    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )


    if patience_count >= PATIENCE:

        print()
        print(
            "[EARLY STOPPING]"
        )

        break


========== EPOCH 2 ==========
batch=0100 loss=0.152443
batch=0200 loss=0.150422
batch=0300 loss=0.149635
batch=0400 loss=0.149569
Train loss      : 0.148393
Accuracy        : 0.664943
Macro Precision : 0.523237
Macro Recall    : 0.554304
Macro F1        : 0.518702
Weighted F1     : 0.604698
[BEST] epoch=2 macro_f1=0.518702

========== EPOCH 3 ==========
batch=0100 loss=0.141289
batch=0200 loss=0.140838
batch=0300 loss=0.140834
batch=0400 loss=0.139938
Train loss      : 0.140618
Accuracy        : 0.520070
Macro Precision : 0.561971
Macro Recall    : 0.443998
Macro F1        : 0.404980
Weighted F1     : 0.448643
[NO IMPROVEMENT] 1/3

========== EPOCH 4 ==========
batch=0100 loss=0.136608
batch=0200 loss=0.137356
batch=0300 loss=0.137109
batch=0400 loss=0.136473
Train loss      : 0.135915
Accuracy        : 0.764370
Macro Precision : 0.631675
Macro Recall    : 0.627023
Macro F1        : 0.588022
Weighted F1     : 0.708751
[BEST] epoch=4 macro_f1=0.588022

========== EPOCH 5 ==========
bat

In [18]:
print()
print("=== TRAINING COMPLETE ===")
print("Best epoch   :", best_epoch)
print(
    "Best Macro F1:",
    f"{best_macro_f1:.6f}"
)
print("Best model   :", BEST_MODEL_PATH)


=== TRAINING COMPLETE ===
Best epoch   : 13
Best Macro F1: 0.663038
Best model   : /home/geonug/kdt-linux/ML/models/network/dnn_top40_v4_best.pt


In [19]:
from sklearn.metrics import confusion_matrix


checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False,
)

best_model = NetworkThreatDNN().to(
    device
)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_model.eval()


print(
    "Loaded epoch:",
    checkpoint["epoch"]
)

print(
    "Dev Macro F1:",
    checkpoint["metrics"]["macro_f1"]
)

Loaded epoch: 13
Dev Macro F1: 0.6630381655347001


In [20]:
@torch.no_grad()
def evaluate_full_parquet(
    model,
    parquet_path,
    features,
    scaler,
    device,
    batch_size=100_000,
):

    parquet_file = pq.ParquetFile(
        parquet_path
    )

    y_true_parts = []
    y_pred_parts = []

    rows_seen = 0

    start = time.perf_counter()


    for batch_id, batch in enumerate(
        parquet_file.iter_batches(
            batch_size=batch_size,
            columns=features + ["label_id"],
        ),
        start=1,
    ):

        df = batch.to_pandas()

        X = df[
            features
        ].to_numpy(
            dtype=np.float32,
            copy=True,
        )

        y = df[
            "label_id"
        ].to_numpy(
            dtype=np.int64,
            copy=True,
        )

        del df, batch


        X = preprocess_batch(
            X,
            scaler,
        )


        X_tensor = torch.from_numpy(
            X
        ).to(device)


        logits = model(
            X_tensor
        )

        pred = torch.argmax(
            logits,
            dim=1,
        ).cpu().numpy()


        y_true_parts.append(y)
        y_pred_parts.append(pred)


        rows_seen += len(y)


        print(
            f"[{batch_id:02d}] "
            f"rows={rows_seen:,}"
        )


        del X
        del X_tensor
        del logits
        del pred

        gc.collect()


    y_true = np.concatenate(
        y_true_parts
    )

    y_pred = np.concatenate(
        y_pred_parts
    )


    elapsed = (
        time.perf_counter()
        - start
    )


    metrics = {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),

        "rows":
            len(y_true),

        "elapsed_seconds":
            elapsed,
    }


    return (
        metrics,
        y_true,
        y_pred,
    )

In [21]:
full_metrics, y_valid_full, y_pred_full = (
    evaluate_full_parquet(
        model=best_model,
        parquet_path=VALID_PATH,
        features=TOP40_FEATURES,
        scaler=scaler,
        device=device,
        batch_size=100_000,
    )
)


print()
print(
    "=== DNN FP32 FULL VALID ==="
)

for key, value in full_metrics.items():

    if isinstance(
        value,
        float,
    ):
        print(
            f"{key:20s}: {value:.6f}"
        )

    else:
        print(
            f"{key:20s}: {value}"
        )

[01] rows=100,000
[02] rows=200,000
[03] rows=300,000
[04] rows=400,000
[05] rows=500,000
[06] rows=600,000
[07] rows=700,000
[08] rows=800,000
[09] rows=900,000
[10] rows=1,000,000
[11] rows=1,100,000
[12] rows=1,200,000
[13] rows=1,300,000
[14] rows=1,400,000
[15] rows=1,500,000
[16] rows=1,600,000
[17] rows=1,700,000
[18] rows=1,800,000
[19] rows=1,900,000
[20] rows=2,000,000
[21] rows=2,100,000
[22] rows=2,200,000
[23] rows=2,300,000
[24] rows=2,390,002

=== DNN FP32 FULL VALID ===
accuracy            : 0.984179
macro_precision     : 0.717727
macro_recall        : 0.704297
macro_f1            : 0.698647
weighted_f1         : 0.981440
rows                : 2390002
elapsed_seconds     : 9.387769


In [22]:
from sklearn.metrics import classification_report


report = classification_report(
    y_valid_full,
    y_pred_full,
    labels=list(range(15)),
    output_dict=True,
    zero_division=0,
)


report_df = (
    pd.DataFrame(report)
    .T
)


print(
    report_df.to_string()
)

              precision    recall  f1-score       support
0              0.990298  0.994464  0.992377  2.012773e+06
1              0.998064  0.998778  0.998421  4.336800e+04
2              0.000000  0.000000  0.000000  9.800000e+01
3              1.000000  0.392857  0.564103  2.800000e+01
4              0.999610  1.000000  0.999805  1.049660e+05
5              0.767442  0.985075  0.862745  2.680000e+02
6              0.982629  0.997009  0.989767  8.658300e+04
7              0.993207  0.996916  0.995058  6.160000e+03
8              0.999305  0.999870  0.999587  6.902400e+04
9              0.000000  0.000000  0.000000  6.882000e+03
10             0.941176  0.983240  0.961749  1.611000e+03
11             0.733467  0.999684  0.846130  1.900500e+04
12             0.364733  0.217317  0.272357  2.462300e+04
13             0.000000  0.000000  0.000000  1.300000e+01
14             0.995972  0.999247  0.997607  1.460000e+04
accuracy       0.984179  0.984179  0.984179  9.841787e-01
macro avg     

In [23]:
report_df.to_csv(
    RESULT_DIR
    / "dnn_fp32_full_valid_classification_report.csv"
)


pd.DataFrame(
    [full_metrics]
).to_csv(
    RESULT_DIR
    / "dnn_fp32_full_valid_metrics.csv",
    index=False,
)

In [24]:
cm = confusion_matrix(
    y_valid_full,
    y_pred_full,
    labels=list(range(15)),
)


cm_df = pd.DataFrame(
    cm,
    index=range(15),
    columns=range(15),
)


cm_df.to_csv(
    RESULT_DIR
    / "dnn_fp32_full_valid_confusion_matrix.csv"
)

In [25]:
print("PyTorch:", torch.__version__)

print(
    "Quantization engines:",
    torch.backends.quantized.supported_engines
)

print(
    "Current engine:",
    torch.backends.quantized.engine
)

PyTorch: 2.14.1+cu130
Quantization engines: ['qnnpack', 'onednn', 'x86', 'fbgemm']
Current engine: x86


In [26]:
fp32_cpu_model = NetworkThreatDNN()

fp32_cpu_model.load_state_dict(
    checkpoint["model_state_dict"]
)

fp32_cpu_model.eval()

NetworkThreatDNN(
  (network): Sequential(
    (0): Linear(in_features=40, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): ReLU()
    (10): Linear(in_features=64, out_features=15, bias=True)
  )
)

In [27]:
import torch.ao.quantization


int8_model = (
    torch.ao.quantization.quantize_dynamic(
        fp32_cpu_model,
        {
            nn.Linear,
        },
        dtype=torch.qint8,
    )
)

int8_model.eval()

print(int8_model)

NetworkThreatDNN(
  (network): Sequential(
    (0): DynamicQuantizedLinear(in_features=40, out_features=128, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): DynamicQuantizedLinear(in_features=128, out_features=128, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): DynamicQuantizedLinear(in_features=128, out_features=64, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
    (9): ReLU()
    (10): DynamicQuantizedLinear(in_features=64, out_features=15, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
  )
)


/tmp/ipykernel_2138337/1101292642.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.quantize_dynamic(
/home/geonug/kdt-linux/ML/.venv/lib/python3.12/site-packages/torch/ao/nn/quantized/modules/utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtyp

In [28]:
INT8_MODEL_PATH = (
    MODEL_DIR
    / "dnn_top40_v4_dynamic_int8.pt"
)


torch.save(
    {
        "model_state_dict":
            int8_model.state_dict(),

        "top40_features":
            TOP40_FEATURES,

        "source_epoch":
            checkpoint["epoch"],

        "quantization":
            "dynamic_int8_linear",
    },
    INT8_MODEL_PATH,
)


print(
    "Saved:",
    INT8_MODEL_PATH
)

Saved: /home/geonug/kdt-linux/ML/models/network/dnn_top40_v4_dynamic_int8.pt


In [29]:
FP32_MODEL_PATH = (
    MODEL_DIR
    / "dnn_top40_v4_fp32_state.pt"
)


torch.save(
    fp32_cpu_model.state_dict(),
    FP32_MODEL_PATH,
)


fp32_mb = (
    FP32_MODEL_PATH.stat().st_size
    / 1024**2
)

int8_mb = (
    INT8_MODEL_PATH.stat().st_size
    / 1024**2
)


print(
    f"FP32 : {fp32_mb:.4f} MB"
)

print(
    f"INT8 : {int8_mb:.4f} MB"
)

print(
    f"Reduction : "
    f"{(1 - int8_mb/fp32_mb)*100:.2f}%"
)

FP32 : 0.1287 MB
INT8 : 0.0447 MB
Reduction : 65.27%


In [30]:
cpu_device = torch.device(
    "cpu"
)


int8_metrics, y_int8, pred_int8 = (
    evaluate_full_parquet(
        model=int8_model,
        parquet_path=VALID_PATH,
        features=TOP40_FEATURES,
        scaler=scaler,
        device=cpu_device,
        batch_size=100_000,
    )
)


print()
print(
    "=== DNN INT8 FULL VALID ==="
)

for key, value in int8_metrics.items():

    if isinstance(value, float):
        print(
            f"{key:20s}: {value:.6f}"
        )

    else:
        print(
            f"{key:20s}: {value}"
        )

[01] rows=100,000
[02] rows=200,000
[03] rows=300,000
[04] rows=400,000
[05] rows=500,000
[06] rows=600,000
[07] rows=700,000
[08] rows=800,000
[09] rows=900,000
[10] rows=1,000,000
[11] rows=1,100,000
[12] rows=1,200,000
[13] rows=1,300,000
[14] rows=1,400,000
[15] rows=1,500,000
[16] rows=1,600,000
[17] rows=1,700,000
[18] rows=1,800,000
[19] rows=1,900,000
[20] rows=2,000,000
[21] rows=2,100,000
[22] rows=2,200,000
[23] rows=2,300,000
[24] rows=2,390,002

=== DNN INT8 FULL VALID ===
accuracy            : 0.878532
macro_precision     : 0.306322
macro_recall        : 0.346922
macro_f1            : 0.286261
weighted_f1         : 0.865403
rows                : 2390002
elapsed_seconds     : 8.002386


In [31]:
int8_report = classification_report(
    y_int8,
    pred_int8,
    labels=list(range(15)),
    output_dict=True,
    zero_division=0,
)

int8_report_df = (
    pd.DataFrame(int8_report)
    .T
)

print(
    int8_report_df.to_string()
)

              precision    recall  f1-score       support
0              0.928210  0.955587  0.941700  2.012773e+06
1              0.428259  0.492160  0.457991  4.336800e+04
2              0.053485  0.336735  0.092308  9.800000e+01
3              0.000843  0.285714  0.001681  2.800000e+01
4              0.985488  0.987929  0.986707  1.049660e+05
5              0.763006  0.985075  0.859935  2.680000e+02
6              0.492952  0.531502  0.511501  8.658300e+04
7              0.047385  0.349838  0.083464  6.160000e+03
8              0.439408  0.014183  0.027480  6.902400e+04
9              0.000000  0.000000  0.000000  6.882000e+03
10             0.304709  0.204842  0.244989  1.611000e+03
11             0.000000  0.000000  0.000000  1.900500e+04
12             0.151089  0.060269  0.086166  2.462300e+04
13             0.000000  0.000000  0.000000  1.300000e+01
14             0.000000  0.000000  0.000000  1.460000e+04
accuracy       0.878532  0.878532  0.878532  8.785323e-01
macro avg     

In [32]:
print(
    "Prediction agreement:",
    f"{np.mean(y_pred_full == pred_int8):.6f}"
)

print(
    "FP32 Macro F1:",
    f"{full_metrics['macro_f1']:.6f}"
)

print(
    "INT8 Macro F1:",
    f"{int8_metrics['macro_f1']:.6f}"
)

print(
    "Macro F1 delta:",
    f"{int8_metrics['macro_f1'] - full_metrics['macro_f1']:+.6f}"
)

Prediction agreement: 0.885160
FP32 Macro F1: 0.698647
INT8 Macro F1: 0.286261
Macro F1 delta: -0.412385


In [33]:
try:
    import torchao

    print("torchao:", torchao.__version__)
    print("torchao available")

except ImportError:
    print("torchao not installed")

torchao not installed


In [34]:
!uv add torchao

Resolved 61 packages in 385ms                                        
Prepared 2 packages in 4.25s                                                 torchao              ------------------------------ 3.20 MiB/3.22 MiB           
Uninstalled 1 package in 4ms
Installed 2 packages in 47ms                                
 ~ ml==0.1.0 (from file:///home/geonug/kdt-linux/ML)
 + torchao==0.18.0


In [1]:
import torch
import torchao

print("torch   :", torch.__version__)
print("torchao :", torchao.__version__)
print("CUDA    :", torch.version.cuda)

/home/geonug/kdt-linux/ML/.venv/lib/python3.12/site-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W1006 14:22:15.983000 2148027 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


torch   : 2.14.1+cu130
torchao : 0.18.0
CUDA    : 13.0


In [2]:
from torchao.quantization import quantize_

print("quantize_ import OK")

quantize_ import OK


In [3]:
import torch
import torchao

from torchao.quantization import quantize_

print("torch   :", torch.__version__)
print("torchao :", torchao.__version__)
print("CUDA    :", torch.version.cuda)
print("quantize_ import OK")

torch   : 2.14.1+cu130
torchao : 0.18.0
CUDA    : 13.0
quantize_ import OK


In [4]:
import torchao.quantization as tq

configs = [
    name
    for name in dir(tq)
    if "Config" in name
]

print("\n".join(sorted(configs)))

AOPerModuleConfig
Float8DynamicActivationFloat8WeightConfig
Float8DynamicActivationInt4WeightConfig
Float8MMConfig
Float8StaticActivationFloat8WeightConfig
Float8WeightOnlyConfig
FqnToConfig
Int4WeightOnlyConfig
Int8DynamicActivationInt8WeightConfig
Int8DynamicActivationIntxWeightConfig
Int8StaticActivationInt8WeightConfig
Int8WeightOnlyConfig
IntxWeightOnlyConfig
ModuleFqnToConfig


In [5]:
import copy
import torch

from torchao.quantization import (
    Int8DynamicActivationInt8WeightConfig,
    quantize_,
)

cpu_device = torch.device("cpu")

# 기존 best FP32 checkpoint에서 새 모델 생성
torchao_int8_model = NetworkThreatDNN().to(cpu_device)

torchao_int8_model.load_state_dict(
    checkpoint["model_state_dict"]
)

torchao_int8_model.eval()

# 양자화 전 sanity check
print(
    "FP32 parameters:",
    sum(p.numel() for p in torchao_int8_model.parameters())
)

NameError: name 'NetworkThreatDNN' is not defined

In [6]:
import torch
import torch.nn as nn


class NetworkThreatDNN(nn.Module):

    def __init__(
        self,
        input_dim=40,
        num_classes=15,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.network(x)


print("NetworkThreatDNN defined")

NetworkThreatDNN defined


In [7]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/home/geonug/kdt-linux/ML"
)

BEST_MODEL_PATH = (
    PROJECT_ROOT
    / "models/network/dnn_top40_v4_best.pt"
)

cpu_device = torch.device("cpu")


checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=cpu_device,
    weights_only=False,
)


print("Best epoch:", checkpoint["epoch"])
print(
    "Dev Macro F1:",
    checkpoint["metrics"]["macro_f1"]
)

Best epoch: 13
Dev Macro F1: 0.6630381655347001


In [8]:
fp32_cpu_model = NetworkThreatDNN().to(
    cpu_device
)

fp32_cpu_model.load_state_dict(
    checkpoint["model_state_dict"]
)

fp32_cpu_model.eval()

print("FP32 model restored")

FP32 model restored


In [9]:
from torchao.quantization import (
    Int8DynamicActivationInt8WeightConfig,
    quantize_,
)


torchao_int8_model = NetworkThreatDNN().to(
    cpu_device
)

torchao_int8_model.load_state_dict(
    checkpoint["model_state_dict"]
)

torchao_int8_model.eval()


config = (
    Int8DynamicActivationInt8WeightConfig()
)

quantize_(
    torchao_int8_model,
    config,
)

torchao_int8_model.eval()

print("TorchAO INT8 applied")
print(torchao_int8_model)

TorchAO INT8 applied
NetworkThreatDNN(
  (network): Sequential(
    (0): Linear(in_features=40, out_features=128, bias=True, weight=Tensor: <class 'torchao.quantization.Int8Tensor'>)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=128, bias=True, weight=Tensor: <class 'torchao.quantization.Int8Tensor'>)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True, weight=Tensor: <class 'torchao.quantization.Int8Tensor'>)
    (9): ReLU()
    (10): Linear(in_features=64, out_features=15, bias=True, weight=Tensor: <class 'torchao.quantization.Int8Tensor'>)
  )
)


In [10]:
from pathlib import Path
import gc
import joblib
import numpy as np
import pandas as pd
import torch


PROJECT_ROOT = Path(
    "/home/geonug/kdt-linux/ML"
)

TOP40_PATH = (
    PROJECT_ROOT
    / "results/feature_selection/top40_features.csv"
)

VALID_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v4/validation/valid_v2_top40.parquet"
)

SCALER_PATH = (
    PROJECT_ROOT
    / "models/network/dnn_top40_v4_scaler.joblib"
)


TOP40_FEATURES = (
    pd.read_csv(TOP40_PATH)["feature"]
    .astype(str)
    .tolist()
)

scaler = joblib.load(
    SCALER_PATH
)


def preprocess_batch(
    X,
    scaler,
):
    X = np.asarray(
        X,
        dtype=np.float32,
    )

    X[~np.isfinite(X)] = np.nan

    means = scaler.mean_.astype(
        np.float32
    )

    rows, cols = np.where(
        np.isnan(X)
    )

    if len(rows) > 0:
        X[rows, cols] = means[cols]

    X = scaler.transform(
        X
    ).astype(
        np.float32,
        copy=False,
    )

    return X


print("Features:", len(TOP40_FEATURES))
print("Scaler features:", scaler.n_features_in_)

Features: 40
Scaler features: 40


In [11]:
import pyarrow.parquet as pq


pf = pq.ParquetFile(
    VALID_PATH
)

batch = next(
    pf.iter_batches(
        batch_size=8192,
        columns=TOP40_FEATURES + ["label_id"],
    )
)

sanity_df = batch.to_pandas()

print(
    "Sanity shape:",
    sanity_df.shape
)

Sanity shape: (8192, 41)


In [12]:
X_sanity = (
    sanity_df[
        TOP40_FEATURES
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)

y_sanity = (
    sanity_df["label_id"]
    .to_numpy(
        dtype=np.int64,
        copy=True,
    )
)

del sanity_df, batch
gc.collect()


X_sanity = preprocess_batch(
    X_sanity,
    scaler,
)

assert np.isfinite(
    X_sanity
).all()


X_tensor = torch.from_numpy(
    X_sanity
)


fp32_cpu_model.eval()
torchao_int8_model.eval()


with torch.inference_mode():

    fp32_logits = fp32_cpu_model(
        X_tensor
    )

    int8_logits = torchao_int8_model(
        X_tensor
    )


    fp32_pred = (
        fp32_logits
        .argmax(dim=1)
        .cpu()
        .numpy()
    )

    int8_pred = (
        int8_logits
        .argmax(dim=1)
        .cpu()
        .numpy()
    )


agreement = np.mean(
    fp32_pred == int8_pred
)


print(
    f"Samples              : {len(fp32_pred):,}"
)

print(
    f"Prediction agreement : {agreement:.6f}"
)

print(
    "Changed predictions  :",
    np.sum(
        fp32_pred != int8_pred
    ),
)

print(
    f"Changed ratio        : "
    f"{1 - agreement:.6f}"
)

Samples              : 8,192
Prediction agreement : 0.999878
Changed predictions  : 1
Changed ratio        : 0.000122


In [13]:
import time
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


@torch.inference_mode()
def evaluate_full_parquet(
    model,
    parquet_path,
    features,
    scaler,
    device,
    batch_size=100_000,
):

    model.eval()

    parquet_file = pq.ParquetFile(
        parquet_path
    )

    y_true_parts = []
    y_pred_parts = []

    rows_seen = 0

    start = time.perf_counter()


    for batch_id, batch in enumerate(
        parquet_file.iter_batches(
            batch_size=batch_size,
            columns=features + ["label_id"],
        ),
        start=1,
    ):

        df = batch.to_pandas()

        X = (
            df[features]
            .to_numpy(
                dtype=np.float32,
                copy=True,
            )
        )

        y = (
            df["label_id"]
            .to_numpy(
                dtype=np.int64,
                copy=True,
            )
        )

        del df, batch


        X = preprocess_batch(
            X,
            scaler,
        )


        X_tensor = (
            torch.from_numpy(X)
            .to(device)
        )


        logits = model(
            X_tensor
        )

        pred = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )


        y_true_parts.append(y)
        y_pred_parts.append(pred)

        rows_seen += len(y)


        print(
            f"[{batch_id:02d}] "
            f"rows={rows_seen:,}"
        )


        del X
        del X_tensor
        del logits
        del pred

        gc.collect()


    y_true = np.concatenate(
        y_true_parts
    )

    y_pred = np.concatenate(
        y_pred_parts
    )


    elapsed = (
        time.perf_counter()
        - start
    )


    metrics = {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),

        "macro_precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "macro_recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "rows": len(y_true),

        "elapsed_seconds": elapsed,
    }


    return metrics, y_true, y_pred

In [14]:
torchao_metrics, y_torchao, pred_torchao = (
    evaluate_full_parquet(
        model=torchao_int8_model,
        parquet_path=VALID_PATH,
        features=TOP40_FEATURES,
        scaler=scaler,
        device=torch.device("cpu"),
        batch_size=100_000,
    )
)


print()
print("=== TORCHAO INT8 FULL VALID ===")

for key, value in torchao_metrics.items():

    if isinstance(value, float):
        print(
            f"{key:20s}: {value:.6f}"
        )
    else:
        print(
            f"{key:20s}: {value}"
        )

[01] rows=100,000
[02] rows=200,000
[03] rows=300,000
[04] rows=400,000
[05] rows=500,000
[06] rows=600,000
[07] rows=700,000
[08] rows=800,000
[09] rows=900,000
[10] rows=1,000,000
[11] rows=1,100,000
[12] rows=1,200,000
[13] rows=1,300,000
[14] rows=1,400,000
[15] rows=1,500,000
[16] rows=1,600,000
[17] rows=1,700,000
[18] rows=1,800,000
[19] rows=1,900,000
[20] rows=2,000,000
[21] rows=2,100,000
[22] rows=2,200,000
[23] rows=2,300,000
[24] rows=2,390,002

=== TORCHAO INT8 FULL VALID ===
accuracy            : 0.984110
macro_precision     : 0.718634
macro_recall        : 0.703598
macro_f1            : 0.698836
weighted_f1         : 0.981409
rows                : 2390002
elapsed_seconds     : 20.058957


In [15]:
FP32_METRICS = {
    "accuracy": 0.984179,
    "macro_precision": 0.717727,
    "macro_recall": 0.704297,
    "macro_f1": 0.698647,
    "weighted_f1": 0.981440,
}


print()
print("=== FP32 → TORCHAO INT8 DELTA ===")

for key in FP32_METRICS:

    fp32_value = FP32_METRICS[key]
    int8_value = torchao_metrics[key]

    print(
        f"{key:16s} "
        f"FP32={fp32_value:.6f}  "
        f"INT8={int8_value:.6f}  "
        f"Δ={int8_value - fp32_value:+.6f}"
    )


=== FP32 → TORCHAO INT8 DELTA ===
accuracy         FP32=0.984179  INT8=0.984110  Δ=-0.000069
macro_precision  FP32=0.717727  INT8=0.718634  Δ=+0.000907
macro_recall     FP32=0.704297  INT8=0.703598  Δ=-0.000699
macro_f1         FP32=0.698647  INT8=0.698836  Δ=+0.000189
weighted_f1      FP32=0.981440  INT8=0.981409  Δ=-0.000031


In [16]:
torchao_report = classification_report(
    y_torchao,
    pred_torchao,
    labels=list(range(15)),
    output_dict=True,
    zero_division=0,
)


torchao_report_df = (
    pd.DataFrame(torchao_report)
    .T
)


print(
    torchao_report_df.to_string()
)

              precision    recall  f1-score       support
0              0.989713  0.994949  0.992324  2.012773e+06
1              0.999284  0.997279  0.998280  4.336800e+04
2              0.000000  0.000000  0.000000  9.800000e+01
3              1.000000  0.392857  0.564103  2.800000e+01
4              0.999600  1.000000  0.999800  1.049660e+05
5              0.767442  0.985075  0.862745  2.680000e+02
6              0.995955  0.984004  0.989943  8.658300e+04
7              0.993381  0.998864  0.996115  6.160000e+03
8              0.999493  0.999870  0.999681  6.902400e+04
9              0.000000  0.000000  0.000000  6.882000e+03
10             0.943981  0.983240  0.963211  1.611000e+03
11             0.733495  0.999684  0.846149  1.900500e+04
12             0.361260  0.218901  0.272615  2.462300e+04
13             0.000000  0.000000  0.000000  1.300000e+01
14             0.995904  0.999247  0.997573  1.460000e+04
accuracy       0.984110  0.984110  0.984110  9.841105e-01
macro avg     

In [17]:
RESULT_DIR = (
    PROJECT_ROOT
    / "results/dnn_top40_v4"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


pd.DataFrame(
    [torchao_metrics]
).to_csv(
    RESULT_DIR
    / "dnn_torchao_int8_full_valid_metrics.csv",
    index=False,
)


torchao_report_df.to_csv(
    RESULT_DIR
    / "dnn_torchao_int8_full_valid_classification_report.csv"
)


cm = confusion_matrix(
    y_torchao,
    pred_torchao,
    labels=list(range(15)),
)


pd.DataFrame(
    cm,
    index=range(15),
    columns=range(15),
).to_csv(
    RESULT_DIR
    / "dnn_torchao_int8_full_valid_confusion_matrix.csv"
)


print("Results saved")

Results saved
